# Exploratory Data Analysis: Global Literacy Rates

## Project Overview

This notebook explores the Global Literacy dataset from Our World in Data.

The dataset contains historical and contemporary estimates of literacy rates across countries and regions, spanning several centuries. Before using the data in a larger analysis, it is important to understand its structure, coverage, quality, and limitations.

This notebook focuses exclusively on exploratory data analysis (EDA) of the literacy dataset.

---

## Objectives

The goals of this analysis are to:

- Understand how literacy is defined and measured.
- Examine the structure of the dataset.
- Assess data quality and completeness.
- Investigate geographic and temporal coverage.
- Identify potential limitations and caveats.
- Generate meaningful analytical questions for future research.

---

## About the Dataset

### Definition of Literacy

According to UNESCO, literacy is defined as:

> The ability of a person aged 15 years and older to read, understand, and write a short, simple statement about their everyday life.

The literacy rate represents the percentage of the adult population (15+) that meets this definition.

---

## Why This Dataset Matters

Literacy is a foundational indicator of educational development and human capital. Understanding literacy trends can provide insight into broader social and economic outcomes, including education access, workforce development, and poverty reduction.

---

## EDA Roadmap

This notebook follows a structured exploratory data analysis process:

### 1. Documentation Review

- Review dataset metadata
- Understand variables and definitions
- Identify known limitations

### 2. Data Inspection

- Examine dataset structure
- Review column data types
- Check missing values and duplicates

### 3. Coverage Analysis

- Explore country coverage
- Explore year coverage
- Identify data availability patterns

### 4. Question Generation

- Identify meaningful analytical questions
- Assess suitability for future research projects

### 5. Summary of Findings

- Document key observations
- Identify next steps for analysis

## **questions**

- how has the global distribution of literacy rates changed overtime
- how does current literacy rate vary geographically across countries
    - cloropleh map


# ideas
- run a simple regression model of literacy rate and illitracy rate

### file imports And Data

In [275]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [276]:
df = pd.read_csv('literate-and-illiterate-world-population.csv')

# EDA

In [277]:
df.head()

,Entity,Code,Year,Illiterate,Literate
0,Afghanistan,AFG,1950,97.000000,3.00000
1,Afghanistan,AFG,1979,82.000000,18.00000
2,Afghanistan,AFG,2011,69.000000,31.00000
3,Afghanistan,AFG,2015,66.246155,33.75384
4,Afghanistan,AFG,2021,63.000000,37.00000


the 2 columns over here 'Illiterate' and 'Literate' are based on percentage

In [278]:
df.columns.to_frame(index=False)

,0
0,Entity
1,Code
2,Year
3,Illiterate
4,Literate


In [279]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Entity      2059 non-null   str    
 1   Code        1687 non-null   str    
 2   Year        2059 non-null   int64  
 3   Illiterate  2059 non-null   float64
 4   Literate    2059 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 80.6 KB


In [280]:
df.describe()

,Year,Illiterate,Literate
count,2059.000000,2059.000000,2059.000000
mean,1986.084507,27.421922,72.578078
std,64.008447,24.485748,24.485748
min,1475.000000,0.000000,0.000000
25%,1983.000000,6.704529,57.981690
50%,2002.000000,21.000000,79.000000
75%,2013.000000,42.018310,93.295470
max,2023.000000,100.000000,100.000000


In [281]:
df.shape

(2059, 5)

2059 rows and 5 cols

### Find missing values and Null values

In [282]:
df.isna().sum()

Entity          0
Code          372
Year            0
Illiterate      0
Literate        0
dtype: int64

In [283]:
df.isnull().sum()

Entity          0
Code          372
Year            0
Illiterate      0
Literate        0
dtype: int64

## investigating missing values

interesting there are 372 null values and missing values in the column "code" which is i suppose the code per countries

In [284]:
missing_code = df[df['Code'].isna()]

pd.Series(missing_code['Entity'].unique())

0                      Central and Southern Asia (SDG)
1                 Eastern and South-Eastern Asia (SDG)
2                    Europe and Northern America (SDG)
3                Latin America and the Caribbean (SDG)
4                    Middle East and North Africa (WB)
5                              Middle-income countries
6               Northern Africa and Western Asia (SDG)
7    Oceania (excluding Australia and New Zealand) ...
8                             Sub-Saharan Africa (SDG)
9              Western Europe (Buringh and van Zanden)
dtype: str

## further reasearch needed on Agrigated enties
upon investigating , i discovered that entities are not countires and the entities column baiscally have agrigated regions

In [285]:
missing_code['Entity'].value_counts()

Entity
Latin America and the Caribbean (SDG)                  50
Middle East and North Africa (WB)                      50
Central and Southern Asia (SDG)                        49
Middle-income countries                                49
Eastern and South-Eastern Asia (SDG)                   48
Northern Africa and Western Asia (SDG)                 48
Sub-Saharan Africa (SDG)                               40
Oceania (excluding Australia and New Zealand) (SDG)    25
Europe and Northern America (SDG)                       9
Western Europe (Buringh and van Zanden)                 4
Name: count, dtype: int64

count of missing values per entity that has missing values

In [286]:
# groupby Entity where first year is 'min' , latest_year 'max' and count of rows
missing_code.groupby('Entity')['Year'].agg(
    First_year = 'min',
    Latest_year = 'max',
    Observations = 'count',
).sort_values('First_year',)

,First_year,Latest_year,Observations
Entity,,,
Western Europe (Buringh and van Zanden),1475,1750,4
Latin America and the Caribbean (SDG),1974,2023,50
Middle East and North Africa (WB),1974,2023,50
Middle-income countries,1975,2023,49
Central and Southern Asia (SDG),1975,2023,49
Northern Africa and Western Asia (SDG),1976,2023,48
Eastern and South-Eastern Asia (SDG),1976,2023,48
Sub-Saharan Africa (SDG),1984,2023,40
Oceania (excluding Australia and New Zealand) (SDG),1994,2021,25


#### min-max years of said entity

### duplicates 

In [287]:
df.duplicated().sum()

np.int64(0)

In [288]:
df.duplicated(subset=['Entity', 'Year']).sum()

np.int64(0)

Verify if the sum of Literate and Illiterate is equals to 100%
note that in documentation it should say everything should be equals to 100%

In [289]:
df[['Literate', 'Illiterate']].describe()

,Literate,Illiterate
count,2059.000000,2059.000000
mean,72.578078,27.421922
std,24.485748,24.485748
min,0.000000,0.000000
25%,57.981690,6.704529
50%,79.000000,21.000000
75%,93.295470,42.018310
max,100.000000,100.000000


In [290]:
(df['Literate'] + df['Illiterate']).head()

0    100.000000
1    100.000000
2    100.000000
3     99.999995
4    100.000000
dtype: float64

there are some rows that does not complete 100%

In [291]:
(df['Literate'] + df['Illiterate']).head()

0    100.000000
1    100.000000
2    100.000000
3     99.999995
4    100.000000
dtype: float64

In [292]:
(df['Literate'] + df['Illiterate']).describe()

count    2059.000000
mean      100.000000
std         0.000001
min        99.999995
25%       100.000000
50%       100.000000
75%       100.000000
max       100.000004
dtype: float64

finding out by how much

In [293]:
total = df.copy() # 
total['Total'] = df['Literate'] + df['Illiterate'] # add 2 columns


not_100 = total.loc[total['Total'] != 100, ['Entity','Total','Year']] 

'''make a subset of the dataset that is only entity total and year that 
shows rows where Total is not 100  and entity != 100 
i picked as well because maybe something happened that year'''

not_100.head()

,Entity,Total,Year
3,Afghanistan,99.999995,2015
10,Albania,99.999999,2017
22,Angola,99.999997,2015
38,Armenia,100.000003,2022
95,Belize,100.000001,2015


look for entities morethan 100

In [294]:
not_100[not_100['Total'] > 100].sort_values('Total',ascending=False).head()
# sort values more than 100 in desc order so that the largest 1st

,Entity,Total,Year
361,East Asia and Pacific (WB),100.000004,1985
374,East Asia and Pacific (WB),100.000004,1998
839,Latin America and the Caribbean (SDG),100.000004,1981
1155,Middle-income countries,100.000004,2000
1630,South Asia (WB),100.000004,2019


looks like more than 100 is just decimal float problem 
i woulnt convert it to int because sometimes it could messup the values

### investigate less than 100 if there are issues

In [295]:
not_100[not_100['Total'] < 100].sort_values('Total',ascending=True).head()

,Entity,Total,Year
3,Afghanistan,99.999995,2015
1249,Niger,99.999995,2018
376,East Asia and Pacific (WB),99.999996,2000
185,Cambodia,99.999996,2021
861,Latin America and the Caribbean (SDG),99.999996,2003


In [296]:
not_100['Total'].agg(['min','max']).to_frame()

,Total
min,99.999995
max,100.000004


### Check the coverage

In [297]:
# min and max year rows
df.loc[[df['Year'].idxmin(),
        df['Year'].idxmax()]]

,Entity,Code,Year,Illiterate,Literate
84,Belgium,BEL,1475,90.0,10.0
55,Azerbaijan,AZE,2023,0.0,100.0


Earliest year and the latest year

In [298]:
earliest_year = df['Year'].min()
latest_year = df['Year'].max()

earliest_data = df[df['Year'] == earliest_year]
latest_data = df[df['Year'] == latest_year]

In [299]:
# earlest data
earliest_data

,Entity,Code,Year,Illiterate,Literate
84,Belgium,BEL,1475,90.0,10.0
566,France,FRA,1475,94.0,6.0
595,Germany,DEU,1475,91.0,9.0
717,Ireland,IRL,1475,100.0,0.0
725,Italy,ITA,1475,85.0,15.0
1232,Netherlands,NLD,1475,83.0,17.0
1448,Poland,POL,1475,100.0,0.0
1638,Spain,ESP,1475,97.0,3.0
1770,Sweden,SWE,1475,99.0,1.0
1871,United Kingdom,GBR,1475,95.0,5.0


In [300]:
(df['Year'] == earliest_year).sum()

np.int64(11)

### Initial Observation
there are only 11 rows in where the year is 1475

In [301]:
earliest_data['Entity']

84                                      Belgium
566                                      France
595                                     Germany
717                                     Ireland
725                                       Italy
1232                                Netherlands
1448                                     Poland
1638                                      Spain
1770                                     Sweden
1871                             United Kingdom
1980    Western Europe (Buringh and van Zanden)
Name: Entity, dtype: str

### Geographic Distribution of Adult Literacy Rate in supposedly 2023 but 1950 has the most entity
- validate the assumption of countries = having a 3 char code

In [302]:
df.columns.to_frame(index=0)


,0
0,Entity
1,Code
2,Year
3,Illiterate
4,Literate


subsetting year only 2023

In [303]:
df_2023 = df[df['Year'] == 2023].copy()

# verify if year = only 2023
df_2023['Year'].agg(['min','max'])

min    2023
max    2023
Name: Year, dtype: int64

In [304]:
df_2023.head()

,Entity,Code,Year,Illiterate,Literate
55,Azerbaijan,AZE,2023,0.000000,100.00000
62,Bahrain,BHR,2023,2.000000,98.00000
255,Central and Southern Asia (SDG),NaN,2023,23.485687,76.51431
399,East Asia and Pacific (WB),WB_EAP,2023,3.569511,96.43049
452,Eastern and South-Eastern Asia (SDG),NaN,2023,3.329498,96.67050


well looking form here looks like there are aggregated regions that has code
we need to find out which are countries and which are aggregared regions

check how many records we have for 2023

In [305]:
df_2023.shape

(28, 5)

huh, thats weird . only 28 entreess we should investigate that

In [306]:
df_2023['Entity'].unique()

<StringArray>
[                            'Azerbaijan',
                                'Bahrain',
        'Central and Southern Asia (SDG)',
             'East Asia and Pacific (WB)',
   'Eastern and South-Eastern Asia (SDG)',
                            'El Salvador',
           'Europe and Central Asia (WB)',
                                'Georgia',
                                  'India',
                                 'Jordan',
       'Latin America and Caribbean (WB)',
  'Latin America and the Caribbean (SDG)',
                   'Low-income countries',
          'Lower-middle-income countries',
      'Middle East and North Africa (WB)',
                'Middle-income countries',
                                  'Nauru',
 'Northern Africa and Western Asia (SDG)',
                                'Senegal',
                        'South Asia (WB)',
                              'Sri Lanka',
               'Sub-Saharan Africa (SDG)',
                'Sub-Saharan Africa (WB)

# !further investigation needed!
maybe the year 2022 has more data and we can use that for out cholopeth map

In [307]:
df_2022 = df[df['Year'] == 2022].copy()

# verify if year = only 2023
df_2022['Year'].agg(['min','max'])

min    2022
max    2022
Name: Year, dtype: int64

In [308]:
df_2022.shape

(44, 5)

maybe in the modern maps it is just aggregated per region.

# objective : find the year that has the most countries covered since regions are aggregated in modern times

In [309]:
df.columns.to_frame(index=0)

,0
0,Entity
1,Code
2,Year
3,Illiterate
4,Literate


In [310]:
# group by year, count unique enitites, sort descending
(
    df.groupby('Year')['Entity']
    .nunique()
    .sort_values(ascending=0)
    .to_frame()
    .reset_index()
)

,Year,Entity
0,1950,178
1,2011,68
2,2000,60
3,2018,60
4,2016,59
...,...,...
88,1939,1
89,1937,1
90,1936,1
91,1947,1


interesting that some years there are only 1 country this gives more questions than answers

### investigare unique entites with the year of 1950 

In [311]:
df_1950 = df[df['Year'] == 1950].copy()
df_1950

,Entity,Code,Year,Illiterate,Literate
0,Afghanistan,AFG,1950,97.0,3.0
5,Albania,ALB,1950,27.5,72.5
11,Algeria,DZA,1950,82.5,17.5
16,American Samoa,ASM,1950,3.5,96.5
18,Andorra,AND,1950,17.5,82.5
...,...,...,...,...,...
2043,Yemen,YEM,1950,97.0,3.0
2047,Yugoslavia,OWID_YGS,1950,27.5,72.5
2048,Zambia,ZMB,1950,77.5,22.5
2054,Zanzibar,OWID_ZAN,1950,92.5,7.5


find unique Entiries for the year 1950 (output collapsed for easy viewing)

In [312]:
for i in enumerate(df_1950['Entity'].unique(), start = 1):
    print(i)

(1, 'Afghanistan')
(2, 'Albania')
(3, 'Algeria')
(4, 'American Samoa')
(5, 'Andorra')
(6, 'Angola')
(7, 'Antigua and Barbuda')
(8, 'Argentina')
(9, 'Australia')
(10, 'Austria')
(11, 'Bahamas')
(12, 'Bahrain')
(13, 'Barbados')
(14, 'Belgium')
(15, 'Belize')
(16, 'Bermuda')
(17, 'Bhutan')
(18, 'Bolivia')
(19, 'Botswana')
(20, 'Brazil')
(21, 'Brunei')
(22, 'Bulgaria')
(23, 'Cambodia')
(24, 'Cameroon')
(25, 'Canada')
(26, 'Cape Verde')
(27, 'Central African Republic')
(28, 'Channel Islands')
(29, 'Chile')
(30, 'China')
(31, 'Cocos Islands')
(32, 'Colombia')
(33, 'Comoros')
(34, 'Cook Islands')
(35, 'Costa Rica')
(36, 'Cuba')
(37, 'Cyprus')
(38, 'Czechoslovakia')
(39, 'Democratic Republic of Congo')
(40, 'Denmark')
(41, 'Djibouti')
(42, 'Dominican Republic')
(43, 'East Timor')
(44, 'Ecuador')
(45, 'Egypt')
(46, 'El Salvador')
(47, 'Equatorial Guinea')
(48, 'Eritrea')
(49, 'Eswatini')
(50, 'Ethiopia')
(51, 'Falkland Islands')
(52, 'Faroe Islands')
(53, 'Fiji')
(54, 'Finland')
(55, 'France')


looks like 1950 is good candiate but it is still pretty old i have to find a hard geojson file to apply coordiates to this

In [313]:
df_1950.nunique()

Entity        178
Code          178
Year            1
Illiterate     25
Literate       25
dtype: int64

## investigate the 2nd 

In [314]:
df.groupby('Entity')['Year'].agg(['min','max','count'])

,min,max,count
Entity,,,
Afghanistan,1950,2021,5
Albania,1950,2017,6
Algeria,1950,2008,5
American Samoa,1950,1980,2
Andorra,1950,1950,1
...,...,...,...
Yemen,1950,1994,2
Yugoslavia,1931,1950,3
Zambia,1950,2018,6


Which countries are suitable for trend analysis or regression \
meaning that they have long time series

In [315]:
(
    df.groupby('Entity')['Year']
    .agg(['min','max','count'])
    .reset_index()
    .sort_values(by='count', ascending= False)
    .head(10)
)

,Entity,min,max,count
236,World,1820,2023,59
115,Latin America and Caribbean (WB),1974,2023,50
116,Latin America and the Caribbean (SDG),1974,2023,50
139,Middle East and North Africa (WB),1974,2023,50
125,Lower-middle-income countries,1975,2023,49
140,Middle-income countries,1975,2023,49
38,Central and Southern Asia (SDG),1975,2023,49
200,South Asia (WB),1975,2023,49
58,East Asia and Pacific (WB),1976,2023,48
229,Upper-middle-income countries,1976,2023,48


# Inspecting Json files came with the package

In [316]:
import json
from pathlib import Path

metadata_path = Path('literate-and-illiterate-world-population.metadata.json')

with open(metadata_path, 'r', encoding = 'utf-8') as f:
    metadata = json.load(f)

In [317]:
type(metadata)

dict

In [318]:
metadata.keys()

dict_keys(['chart', 'columns', 'dateDownloaded', 'activeFilters'])

so it has charts, date downloaded and active files maybe more exploration will help

check the col first

In [319]:
type(metadata['columns'])

dict

In [320]:
metadata['columns'].keys()

dict_keys(['Illiteracy rate', 'Literacy rate'])

lets check whats inside

In [321]:
for col,info in metadata['columns'].items():
    print(col)
    print(info.keys())
    print()


Illiteracy rate
dict_keys(['titleShort', 'titleLong', 'descriptionShort', 'descriptionKey', 'descriptionProcessing', 'shortUnit', 'unit', 'timespan', 'type', 'owidVariableId', 'shortName', 'lastUpdated', 'nextUpdate', 'citationShort', 'citationLong', 'fullMetadata'])

Literacy rate
dict_keys(['titleShort', 'titleLong', 'descriptionShort', 'descriptionKey', 'descriptionProcessing', 'shortUnit', 'unit', 'timespan', 'type', 'owidVariableId', 'shortName', 'lastUpdated', 'nextUpdate', 'citationShort', 'citationLong', 'fullMetadata'])



lets start inspecting <code>descriptionShort</code>

In [322]:
print('Illiteracy :', metadata['columns']['Illiteracy rate']['descriptionShort'])
print('literacy :', metadata['columns']['Literacy rate']['descriptionShort'])

Illiteracy : Share of adults who cannot read and write a short, simple statement on their everyday life.
literacy : Share of adults who can read and write a simple statement about their everyday life.


well i guess i was asking for that

time Span

In [323]:
print('Illiteracy :', metadata['columns']['Illiteracy rate']['timespan'])
print('literacy :', metadata['columns']['Literacy rate']['timespan'])

Illiteracy : 1475-2023
literacy : 1475-2023


date downloaded

In [324]:
type(metadata['dateDownloaded'])

str

In [325]:
metadata['dateDownloaded']

'2026-06-11'

well i guess i can use this for validations

In [326]:
metadata["activeFilters"]

{}

?